# 第7章: 機械学習

本章では、[Stanford Sentiment Treebank (SST)](https://nlp.stanford.edu/sentiment/) データセットを用い、評判分析器（ポジネガ分類器）を構築する。ここでは処理を簡略化するため、[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されているSSTデータセットを用いる。


## 60. データの入手・整形

GLUEのウェブサイトから[SST-2](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip)データセットを取得せよ。学習データ（`train.tsv`）と検証データ（`dev.tsv`）のぞれぞれについて、ポジティブ (1) とネガティブ (0) の事例数をカウントせよ。

In [1]:
!wget https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
!unzip SST-2.zip

--2026-05-21 13:15:27--  https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.171.22.33, 3.171.22.68, 3.171.22.118, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.171.22.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7439277 (7.1M) [application/zip]
Saving to: ‘SST-2.zip’

SST-2.zip           100%[===================>]   7.09M  --.-KB/s    in 0.09s   

2026-05-21 13:15:28 (77.3 MB/s) - ‘SST-2.zip’ saved [7439277/7439277]

Archive:  SST-2.zip
   creating: SST-2/
  inflating: SST-2/dev.tsv           
   creating: SST-2/original/
  inflating: SST-2/original/README.txt  
  inflating: SST-2/original/SOStr.txt  
  inflating: SST-2/original/STree.txt  
  inflating: SST-2/original/datasetSentences.txt  
  inflating: SST-2/original/datasetSplit.txt  
  inflating: SST-2/original/dictionary.txt  
  inflating: SST-2/original/original_rt_snippets.txt  
  inflating: SST-2/original/sentimen

In [6]:
!head SST-2/train.tsv

sentence	label
hide new secretions from the parental units 	0
contains no wit , only labored gags 	0
that loves its characters and communicates something rather beautiful about human nature 	1
remains utterly satisfied to remain the same throughout 	0
on the worst revenge-of-the-nerds clichés the filmmakers could dredge up 	0
that 's far too tragic to merit such superficial treatment 	0
demonstrates that the director of such hollywood blockbusters as patriot games can still turn out a small , personal film with an emotional wallop . 	1
of saucy 	1
a depressed fifteen-year-old 's suicidal poetry 	0


In [5]:
import pandas as pd

for path in ["SST-2/train.tsv", "SST-2/dev.tsv"]:
    df = pd.read_csv(path, sep="\t")
    print(path)
    print(df["label"].value_counts())

SST-2/train.tsv
label
1    37569
0    29780
Name: count, dtype: int64
SST-2/dev.tsv
label
1    444
0    428
Name: count, dtype: int64


## 61. 特徴ベクトル

Bag of Words (BoW) に基づき、学習データ（`train.tsv`）および検証データ（`dev.tsv`）のテキストを特徴ベクトルに変換したい。ここで、ある事例のテキストの特徴ベクトルは、テキスト中に含まれる単語（スペース区切りのトークン）の出現頻度で構成する。例えば、"too loud , too goofy"というテキストに対応する特徴ベクトルは、以下のような辞書オブジェクトで表現される。

```python
{'too': 2, 'loud': 1, ',': 1, 'goofy': 1}
```

各事例はテキスト、特徴ベクトル、ラベルを格納した辞書オブジェクトでまとめておく。例えば、先ほどの"too loud , too goofy"に対してラベル"0"（ネガティブ）が付与された事例は、以下のオブジェクトで表現される。

```python
{'text': 'too loud , too goofy', 'label': '0', 'feature': {'too': 2, 'loud': 1, ',': 1, 'goofy': 1}}
```

学習データと検証データの各事例を上記のような辞書オブジェクトに変換したうえで、学習データと検証データのそれぞれを、辞書オブジェクトのリストとして表現せよ。さらに、学習データの最初の事例について、正しく特徴ベクトルに変換できたか、目視で確認せよ。

In [7]:
import pandas as pd
from collections import Counter

# TSVファイルの読み込み
train_df = pd.read_csv("SST-2/train.tsv", sep="\t")
dev_df = pd.read_csv("SST-2/dev.tsv", sep="\t")

# 1事例を辞書オブジェクトへ変換する関数
def create_example(row):
    text = row["sentence"]
    label = str(row["label"])

    # スペース区切りでトークン化
    tokens = text.split()

    # Bag of Words
    feature = dict(Counter(tokens))

    return {
        "text": text,
        "label": label,
        "feature": feature
    }

# 学習データ
train_data = [create_example(row) for _, row in train_df.iterrows()]

# 検証データ
dev_data = [create_example(row) for _, row in dev_df.iterrows()]

# 学習データの最初の事例を確認
print(train_data[0])

{'text': 'hide new secretions from the parental units ', 'label': '0', 'feature': {'hide': 1, 'new': 1, 'secretions': 1, 'from': 1, 'the': 1, 'parental': 1, 'units': 1}}


## 62. 学習

61で構築した学習データの特徴ベクトルを用いて、ロジスティック回帰モデルを学習せよ。

In [8]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression


# 特徴ベクトルとラベルを取り出す
X_train = [data["feature"] for data in train_data]
y_train = [data["label"] for data in train_data]

# BoW辞書を数値ベクトルへ変換
vectorizer = DictVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)

# ロジスティック回帰モデル
model = LogisticRegression(max_iter=1000)

# 学習
model.fit(X_train_vec, y_train)

LogisticRegression(max_iter=1000)

## 63. 予測

学習したロジスティック回帰モデルを用い、検証データの先頭の事例のラベル（ポジネガ）を予測せよ。また、予測されたラベルが検証データで付与されていたラベルと一致しているか、確認せよ。

In [10]:
# 検証データの先頭1件を取り出す
example = dev_data[0]

print("文章:")
print(example["text"])

print("\n正解ラベル:")
print(example["label"])

# 特徴量をベクトル化
X_dev_vec = vectorizer.transform([example["feature"]])

# ラベル予測
pred_label = model.predict(X_dev_vec)[0]

print("\n予測ラベル:")
print(pred_label)

文章:
it 's a charming and often affecting journey . 

正解ラベル:
1

予測ラベル:
1


## 64. 条件付き確率

学習したロジスティック回帰モデルを用い、検証データの先頭の事例を各ラベル（ポジネガ）に分類するときの条件付き確率を求めよ。

In [11]:
# 検証データの先頭1件
example = dev_data[0]

# BoW辞書を数値ベクトルへ変換
X_dev_vec = vectorizer.transform([example["feature"]])

# 各ラベルの条件付き確率を求める
probs = model.predict_proba(X_dev_vec)[0]

# ラベルの順番を確認
labels = model.classes_

print("文章:", example["text"])
print("正解ラベル:", example["label"])

for label, prob in zip(labels, probs):
    label_name = "negative" if label == "0" else "positive"
    print(f"ラベル {label} ({label_name}) の確率: {prob:.4f}")

文章: it 's a charming and often affecting journey . 
正解ラベル: 1
ラベル 0 (negative) の確率: 0.0043
ラベル 1 (positive) の確率: 0.9957


## 65. テキストのポジネガの予測

与えられたテキストのポジネガを予測するプログラムを実装せよ。例えば、テキストとして"the worst movie I 've ever seen"を与え、ロジスティック回帰モデルの予測結果を確認せよ。


## 66. 混同行列の作成

学習したロジスティック回帰モデルの検証データにおける混同行列（confusion matrix）を求めよ。

## 67. 精度の計測

学習したロジスティック回帰モデルの正解率、適合率、再現率、F1スコアを、学習データおよび検証データ上で計測せよ。

## 68. 特徴量の重みの確認

学習したロジスティック回帰モデルの中で、重みの高い特徴量トップ20と、重みの低い特徴量トップ20を確認せよ。

## 69. 正則化パラメータの変更

ロジスティック回帰モデルを学習するとき、正則化の係数（ハイパーパラメータ）を調整することで、学習時の適合度合いを制御できる。正則化の係数を変化させながらロジスティック回帰モデルを学習し、検証データ上の正解率を求めよ。実験の結果は、正則化パラメータを横軸、正解率を縦軸としたグラフにまとめよ。